In [1]:
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 160)

In [2]:
ROOT = Path(".").resolve()
INPUT_DIR = ROOT / "solar_installed_inputs"
OUTPUT_DIR = ROOT / "solar_installed_outputs"
COUNTY_GEOJSON = ROOT / "county_layer_for_gui.geojson"

RECENT_LABEL = "2024"

EIA_2021_SOLAR = INPUT_DIR / "3_3_Solar_Y2021.xlsx"
EIA_2021_PLANT = INPUT_DIR / "2___Plant_Y2021.xlsx"

EIA_2024_SOLAR = INPUT_DIR / "3_3_Solar_Y2024.xlsx"
EIA_2024_PLANT = INPUT_DIR / "2___Plant_Y2024.xlsx"

GEM_FILE = INPUT_DIR / "Global-Solar-Power-Tracker-February-2026.xlsx"

OUTPUT_DIR.mkdir(exist_ok=True)

for p in [
    INPUT_DIR,
    OUTPUT_DIR,
    COUNTY_GEOJSON,
    EIA_2021_SOLAR,
    EIA_2021_PLANT,
    EIA_2024_SOLAR,
    EIA_2024_PLANT,
    GEM_FILE,
]:
    print(f"{p.name}: {p.exists()}")

solar_installed_inputs: True
solar_installed_outputs: True
county_layer_for_gui.geojson: True
3_3_Solar_Y2021.xlsx: True
2___Plant_Y2021.xlsx: True
3_3_Solar_Y2024.xlsx: True
2___Plant_Y2024.xlsx: True
Global-Solar-Power-Tracker-February-2026.xlsx: True


In [3]:
def list_excel_sheets(path):
    xls = pd.ExcelFile(path)
    return xls.sheet_names

files_to_check = [
    EIA_2021_SOLAR,
    EIA_2021_PLANT,
    EIA_2024_SOLAR,
    EIA_2024_PLANT,
    GEM_FILE,
]

for f in files_to_check:
    print(f"\n{f.name}")
    try:
        sheets = list_excel_sheets(f)
        for s in sheets:
            print(" -", s)
    except Exception as e:
        print("Could not read sheets:", e)


3_3_Solar_Y2021.xlsx
 - Operable
 - Retired and Canceled

2___Plant_Y2021.xlsx
 - Plant

3_3_Solar_Y2024.xlsx
 - Operable
 - Retired and Canceled

2___Plant_Y2024.xlsx
 - Plant

Global-Solar-Power-Tracker-February-2026.xlsx
 - About
 - Utility-Scale (1 MW+)
 - Distributed (<1 MW)


In [4]:
EIA_2021_SOLAR_SHEET = None
EIA_2021_PLANT_SHEET = None

EIA_2024_SOLAR_SHEET = None
EIA_2024_PLANT_SHEET = None

GEM_SHEET = "Utility-Scale (1 MW+)"

In [5]:
def norm_text(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).lower()).strip()

def find_col(columns, candidates):
    norm_map = {c: norm_text(c) for c in columns}
    candidate_norms = [norm_text(c) for c in candidates]

    for cand in candidate_norms:
        for col, ncol in norm_map.items():
            if ncol == cand:
                return col

    for cand in candidate_norms:
        for col, ncol in norm_map.items():
            if cand in ncol:
                return col
    return None

def choose_sheet(path, preferred_keywords=None):
    preferred_keywords = preferred_keywords or []
    xls = pd.ExcelFile(path)
    sheets = xls.sheet_names

    excluded = ["readme", "cover", "contents", "instruction", "notes", "legend", "layout"]

    for kw in preferred_keywords:
        for s in sheets:
            if kw.lower() in s.lower():
                return s

    filtered = [s for s in sheets if not any(bad in s.lower() for bad in excluded)]
    if filtered:
        return filtered[0]

    return sheets[0]

def read_excel_auto(path, sheet_name=None, preferred_keywords=None):
    if sheet_name is None:
        sheet_name = choose_sheet(path, preferred_keywords=preferred_keywords)
    df = pd.read_excel(path, sheet_name=sheet_name)
    df.columns = [str(c).strip() for c in df.columns]
    return df, sheet_name

def to_numeric_series(s):
    return pd.to_numeric(s, errors="coerce")

def extract_year_series(s):
    if s is None:
        return pd.Series(dtype="Int64")
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce").astype("Int64")
    text = s.astype(str).str.extract(r"((?:19|20)\d{2})")[0]
    return pd.to_numeric(text, errors="coerce").astype("Int64")

def clean_text_series(s):
    return (
        s.astype(str)
        .str.strip()
        .replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
    )

def clean_state_abbrev(s):
    return (
        s.astype(str)
        .str.strip()
        .str.upper()
        .replace({"NAN": pd.NA, "NONE": pd.NA, "": pd.NA})
    )

def standardize_county_name_for_join(s):
    return (
        s.astype(str)
        .str.strip()
        .str.replace(r"\s+County$", "", regex=True)
        .str.replace(r"\s+Parish$", "", regex=True)
        .str.replace(r"\s+Borough$", "", regex=True)
        .str.replace(r"\s+Census Area$", "", regex=True)
        .str.replace(r"\s+Municipality$", "", regex=True)
        .replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
    )

def aggregate_capacity(df, group_cols):
    out = (
        df.groupby(group_cols, dropna=False)
        .agg(
            project_count=("source_project_id", "count"),
            total_capacity_mw=("capacity_mw", "sum"),
            avg_capacity_mw=("capacity_mw", "mean"),
        )
        .reset_index()
        .sort_values("total_capacity_mw", ascending=False)
    )
    return out

def compare_aggregates(left_df, right_df, keys, left_label, right_label):
    merged = left_df.merge(
        right_df,
        on=keys,
        how="outer",
        suffixes=(f"_{left_label}", f"_{right_label}")
    )

    merged[f"capacity_diff_mw_{left_label}_minus_{right_label}"] = (
        merged[f"total_capacity_mw_{left_label}"].fillna(0)
        - merged[f"total_capacity_mw_{right_label}"].fillna(0)
    )

    merged[f"project_count_diff_{left_label}_minus_{right_label}"] = (
        merged[f"project_count_{left_label}"].fillna(0)
        - merged[f"project_count_{right_label}"].fillna(0)
    )

    return merged

In [6]:
counties = gpd.read_file(COUNTY_GEOJSON).to_crs("EPSG:4326").copy()

counties = counties.rename(
    columns={
        "NAME": "county_name_geo",
        "STATE_NAME": "state_name_geo",
        "STUSPS": "state_abbrev_geo",
        "GEOID": "county_fips",
    }
)

counties["county_name_join"] = standardize_county_name_for_join(counties["county_name_geo"])
counties["state_abbrev_geo"] = clean_state_abbrev(counties["state_abbrev_geo"])
counties["state_name_geo"] = clean_text_series(counties["state_name_geo"])

state_lookup = (
    counties[["state_abbrev_geo", "state_name_geo"]]
    .drop_duplicates()
    .rename(columns={"state_abbrev_geo": "state_abbrev", "state_name_geo": "state_name"})
)

print(counties.shape)
display(counties[["county_name_geo", "state_name_geo", "state_abbrev_geo", "county_fips"]].head())

(3235, 17)


,county_name_geo,state_name_geo,state_abbrev_geo,county_fips
0,Houston,Alabama,AL,01069
1,Choctaw,Alabama,AL,01023
2,Russell,Alabama,AL,01113
3,Sussex,Delaware,DE,10005
4,Jackson,Alabama,AL,01071


In [7]:
def score_columns(columns, targets):
    norm_cols = [norm_text(c) for c in columns]
    score = 0
    for t in targets:
        nt = norm_text(t)
        for c in norm_cols:
            if nt == c or nt in c:
                score += 1
                break
    return score

def inspect_excel_candidates(path, expected_cols, max_header_row=6):
    xls = pd.ExcelFile(path)
    results = []

    for sheet in xls.sheet_names:
        for header_row in range(max_header_row + 1):
            try:
                df = pd.read_excel(path, sheet_name=sheet, header=header_row, nrows=5)
                cols = [str(c).strip() for c in df.columns]
                score = score_columns(cols, expected_cols)
                results.append({
                    "sheet": sheet,
                    "header_row": header_row,
                    "score": score,
                    "columns": cols[:20],
                })
            except Exception:
                continue

    return pd.DataFrame(results).sort_values(
        ["score", "sheet", "header_row"], ascending=[False, True, True]
    )

def read_excel_best(path, expected_cols, preferred_sheet_keywords=None, max_header_row=6):
    preferred_sheet_keywords = preferred_sheet_keywords or []
    candidates = inspect_excel_candidates(path, expected_cols, max_header_row=max_header_row).copy()

    if candidates.empty:
        raise ValueError(f"Could not inspect workbook: {path}")

    candidates["preferred"] = False
    for kw in preferred_sheet_keywords:
        candidates.loc[candidates["sheet"].str.lower().str.contains(kw.lower(), na=False), "preferred"] = True

    candidates = candidates.sort_values(
        ["preferred", "score", "sheet", "header_row"],
        ascending=[False, False, True, True]
    )

    best = candidates.iloc[0]
    df = pd.read_excel(path, sheet_name=best["sheet"], header=int(best["header_row"]))
    df.columns = [str(c).strip() for c in df.columns]

    return df, best, candidates

def require_col(columns, candidates, label):
    col = find_col(columns, candidates)
    if col is None:
        raise ValueError(f"Could not find required column for {label}. Available columns: {list(columns)}")
    return col

def standardize_eia_solar(solar_path, plant_path, snapshot_year):
    solar_expected = [
        "Plant Code", "Generator ID", "Status", "Nameplate Capacity (MW)",
        "Net Summer Capacity (MW)", "Operating Year", "Operating Date",
        "Technology", "Energy Source 1"
    ]
    plant_expected = [
        "Plant Code", "Plant Name", "County", "State", "State Name"
    ]

    solar_df, solar_best, solar_candidates = read_excel_best(
        solar_path,
        expected_cols=solar_expected,
        preferred_sheet_keywords=["operable", "solar", "existing"]
    )

    plant_df, plant_best, plant_candidates = read_excel_best(
        plant_path,
        expected_cols=plant_expected,
        preferred_sheet_keywords=["plant"]
    )

    print(f"\nEIA {snapshot_year}")
    print("Solar sheet/header selected:", solar_best["sheet"], "| header row:", solar_best["header_row"], "| score:", solar_best["score"])
    print("Plant sheet/header selected:", plant_best["sheet"], "| header row:", plant_best["header_row"], "| score:", plant_best["score"])

    solar_plant_code_col = require_col(solar_df.columns, ["Plant Code", "Plant ID", "Plant Id"], "solar plant code")
    solar_capacity_col = require_col(
        solar_df.columns,
        ["Nameplate Capacity (MW)", "Net Summer Capacity (MW)", "Summer Capacity (MW)", "Capacity (MW)"],
        "solar capacity"
    )

    solar_status_col = find_col(solar_df.columns, ["Status", "Operational Status", "Generator Status"])
    solar_op_year_col = find_col(solar_df.columns, ["Operating Year", "Operating Date", "Commercial Operation Date", "Current Year"])
    solar_gen_id_col = find_col(solar_df.columns, ["Generator ID", "Generator Id"])
    solar_tech_col = find_col(solar_df.columns, ["Technology", "Prime Mover", "Solar Technology"])
    solar_energy_col = find_col(solar_df.columns, ["Energy Source 1", "Reported Energy Source 1", "Energy Source"])

    plant_plant_code_col = require_col(plant_df.columns, ["Plant Code", "Plant ID", "Plant Id"], "plant plant code")
    plant_name_col = find_col(plant_df.columns, ["Plant Name", "Name"])
    plant_county_col = require_col(plant_df.columns, ["County"], "plant county")
    plant_state_abbrev_col = require_col(plant_df.columns, ["State", "Plant State"], "plant state abbrev")
    plant_state_name_col = find_col(plant_df.columns, ["State Name"])

    print("\nDetected EIA columns:")
    print("  solar plant code:", solar_plant_code_col)
    print("  solar status:", solar_status_col)
    print("  solar capacity:", solar_capacity_col)
    print("  solar op year:", solar_op_year_col)
    print("  solar generator id:", solar_gen_id_col)
    print("  solar technology:", solar_tech_col)
    print("  solar energy:", solar_energy_col)
    print("  plant plant code:", plant_plant_code_col)
    print("  plant name:", plant_name_col)
    print("  plant county:", plant_county_col)
    print("  plant state abbrev:", plant_state_abbrev_col)
    print("  plant state name:", plant_state_name_col)

    # Rename plant columns before merging so there is never a suffix conflict
    plant_merge = plant_df.copy()
    rename_map = {
        plant_plant_code_col: "plant_code_join",
        plant_county_col: "plant_county",
        plant_state_abbrev_col: "plant_state_abbrev",
    }
    if plant_name_col is not None:
        rename_map[plant_name_col] = "plant_name_ref"
    if plant_state_name_col is not None:
        rename_map[plant_state_name_col] = "plant_state_name_ref"

    plant_merge = plant_merge.rename(columns=rename_map)

    plant_keep = [c for c in ["plant_code_join", "plant_name_ref", "plant_county", "plant_state_abbrev", "plant_state_name_ref"] if c in plant_merge.columns]

    merged = solar_df.merge(
        plant_merge[plant_keep].drop_duplicates(),
        left_on=solar_plant_code_col,
        right_on="plant_code_join",
        how="left"
    )

    status_text = merged[solar_status_col].astype(str).str.lower() if solar_status_col else pd.Series("", index=merged.index)
    is_operating = (
        status_text.str.contains(r"operat|existing|^op$", regex=True, na=False)
        if solar_status_col else pd.Series(True, index=merged.index)
    )

    gen_id_part = merged[solar_gen_id_col].astype(str) if solar_gen_id_col else merged.index.astype(str)

    out = pd.DataFrame({
        "source": "EIA",
        "snapshot_label": str(snapshot_year),
        "source_project_id": merged[solar_plant_code_col].astype(str) + "_" + gen_id_part,
        "plant_code": merged[solar_plant_code_col],
        "plant_name": merged["plant_name_ref"] if "plant_name_ref" in merged.columns else pd.NA,
        "county_name": clean_text_series(merged["plant_county"]) if "plant_county" in merged.columns else pd.Series(pd.NA, index=merged.index),
        "state_abbrev": clean_state_abbrev(merged["plant_state_abbrev"]) if "plant_state_abbrev" in merged.columns else pd.Series(pd.NA, index=merged.index),
        "state_name": clean_text_series(merged["plant_state_name_ref"]) if "plant_state_name_ref" in merged.columns else pd.Series(pd.NA, index=merged.index),
        "technology": merged[solar_tech_col] if solar_tech_col else "Solar",
        "energy_source": merged[solar_energy_col] if solar_energy_col else "Solar",
        "status": merged[solar_status_col] if solar_status_col else pd.NA,
        "operating_year": extract_year_series(merged[solar_op_year_col]) if solar_op_year_col else pd.Series(pd.NA, index=merged.index, dtype="Int64"),
        "capacity_mw": to_numeric_series(merged[solar_capacity_col]),
    })

    out = out[is_operating].copy()
    out = out[out["capacity_mw"].notna() & (out["capacity_mw"] > 0)].copy()

    out["county_name_join"] = standardize_county_name_for_join(out["county_name"])

    out = out.merge(
        state_lookup,
        on="state_abbrev",
        how="left",
        suffixes=("", "_lookup")
    )
    out["state_name"] = out["state_name"].fillna(out["state_name_lookup"])
    out = out.drop(columns=[c for c in ["state_name_lookup"] if c in out.columns])

    out = out.merge(
        counties[["state_abbrev_geo", "county_name_join", "county_fips"]]
        .drop_duplicates()
        .rename(columns={"state_abbrev_geo": "state_abbrev"}),
        on=["state_abbrev", "county_name_join"],
        how="left"
    )

    if out["operating_year"].notna().any():
        out = out[out["operating_year"].fillna(snapshot_year) <= snapshot_year].copy()

    return out.reset_index(drop=True), solar_candidates, plant_candidates

In [8]:
eia_2021, eia_2021_solar_candidates, eia_2021_plant_candidates = standardize_eia_solar(
    EIA_2021_SOLAR,
    EIA_2021_PLANT,
    2021,
)

eia_2024, eia_2024_solar_candidates, eia_2024_plant_candidates = standardize_eia_solar(
    EIA_2024_SOLAR,
    EIA_2024_PLANT,
    2024,
)

print("eia_2021 shape:", eia_2021.shape)
print("eia_2024 shape:", eia_2024.shape)

display(eia_2021.head())
display(eia_2024.head())


EIA 2021
Solar sheet/header selected: Operable | header row: 1 | score: 6
Plant sheet/header selected: Plant | header row: 1 | score: 4

Detected EIA columns:
  solar plant code: Plant Code
  solar status: Status
  solar capacity: Nameplate Capacity (MW)
  solar op year: Operating Year
  solar generator id: Generator ID
  solar technology: Technology
  solar energy: None
  plant plant code: Plant Code
  plant name: Plant Name
  plant county: County
  plant state abbrev: State
  plant state name: None

EIA 2024
Solar sheet/header selected: Operable | header row: 1 | score: 6
Plant sheet/header selected: Plant | header row: 1 | score: 4

Detected EIA columns:
  solar plant code: Plant Code
  solar status: Status
  solar capacity: Nameplate Capacity (MW)
  solar op year: Operating Year
  solar generator id: Generator ID
  solar technology: Technology
  solar energy: None
  plant plant code: Plant Code
  plant name: Plant Name
  plant county: County
  plant state abbrev: State
  plant sta

,source,snapshot_label,source_project_id,plant_code,plant_name,county_name,state_abbrev,state_name,technology,energy_source,status,operating_year,capacity_mw,county_name_join,county_fips
0,EIA,2021,645_1,645,Big Bend,Hillsborough,FL,Florida,Solar Photovoltaic,Solar,OP,2017,19.8,Hillsborough,12057
1,EIA,2021,944_12,944,Geneseo,Henry,IL,Illinois,Solar Photovoltaic,Solar,OP,2015,1.2,Henry,17073
2,EIA,2021,960_SOL1,960,North Ninth Street,Ogle,IL,Illinois,Solar Photovoltaic,Solar,OP,2014,0.3,Ogle,17141
3,EIA,2021,1016_10,1016,Butler-Warner Generation Plant,Cumberland,NC,North Carolina,Solar Photovoltaic,Solar,OP,2019,1.0,Cumberland,37051
4,EIA,2021,1172_SO,1172,Osage (IA),Mitchell,IA,Iowa,Solar Photovoltaic,Solar,OP,2016,0.8,Mitchell,19131


,source,snapshot_label,source_project_id,plant_code,plant_name,county_name,state_abbrev,state_name,technology,energy_source,status,operating_year,capacity_mw,county_name_join,county_fips
0,EIA,2024,141_PV-3,141,Agua Fria,Maricopa,AZ,Arizona,Solar Photovoltaic,Solar,OP,2001,0.2,Maricopa,04013
1,EIA,2024,645_1,645,Big Bend,Hillsborough,FL,Florida,Solar Photovoltaic,Solar,OP,2017,19.8,Hillsborough,12057
2,EIA,2024,944_12,944,Geneseo,Henry,IL,Illinois,Solar Photovoltaic,Solar,OP,2015,1.2,Henry,17073
3,EIA,2024,960_SOL1,960,North Ninth Street,Ogle,IL,Illinois,Solar Photovoltaic,Solar,OP,2014,0.3,Ogle,17141
4,EIA,2024,1016_10,1016,Butler-Warner Generation Plant,Cumberland,NC,North Carolina,Solar Photovoltaic,Solar,OP,2019,1.0,Cumberland,37051


In [9]:
def standardize_gem(gem_path, gem_sheet=None):
    gem_df, gem_sheet_used = read_excel_auto(
        gem_path,
        sheet_name=gem_sheet,
        preferred_keywords=["utility", "project", "projects", "tracker", "solar"]
    )

    print("\nGEM sheet used:", gem_sheet_used)
    print("GEM raw shape:", gem_df.shape)
    print("First 40 GEM columns:")
    print(gem_df.columns.tolist()[:40])

    country_col = find_col(gem_df.columns, ["Country/Area", "Country", "Country Area"])
    status_col = find_col(gem_df.columns, ["Status", "Project Status", "Phase Status", "Operating Status"])
    state_col = find_col(gem_df.columns, ["State/Province"])
    if state_col is None:
        state_col = find_col(gem_df.columns, ["Major area (prefecture, district)", "Subregion", "Region"])
    county_col = find_col(gem_df.columns, ["Local area (taluk, county)", "County", "County Name"])
    lat_col = find_col(gem_df.columns, ["Latitude", "Lat"])
    lon_col = find_col(gem_df.columns, ["Longitude", "Lon", "Long"])
    capacity_col = find_col(gem_df.columns, ["Capacity (MW)", "Capacity MW", "Capacity"])
    project_name_col = find_col(gem_df.columns, ["Project Name", "Name", "Plant Name", "Asset Name"])
    online_year_col = find_col(gem_df.columns, ["Start year", "Operating Year", "Commissioning Year", "Operating Date", "Commercial Operation Date"])
    gem_id_col = find_col(gem_df.columns, ["GEM location ID", "Project ID", "GEM ID", "Wiki ID", "Global ID", "ID"])

    print("Detected GEM columns:")
    print("  country:", country_col)
    print("  status:", status_col)
    print("  state:", state_col)
    print("  county:", county_col)
    print("  lat:", lat_col)
    print("  lon:", lon_col)
    print("  capacity:", capacity_col)
    print("  project name:", project_name_col)
    print("  online year:", online_year_col)
    print("  project id:", gem_id_col)

    idx = gem_df.index

    def safe_text(col):
        if col is None:
            return pd.Series(pd.NA, index=idx, dtype="object")
        return clean_text_series(gem_df[col])

    def safe_num(col):
        if col is None:
            return pd.Series(np.nan, index=idx)
        return to_numeric_series(gem_df[col])

    out = pd.DataFrame({
        "source": "GEM",
        "source_project_id": gem_df[gem_id_col].astype(str) if gem_id_col else idx.astype(str),
        "project_name": safe_text(project_name_col),
        "county_name": safe_text(county_col),
        "state_name": safe_text(state_col),
        "state_abbrev": pd.Series(pd.NA, index=idx, dtype="object"),
        "latitude": safe_num(lat_col),
        "longitude": safe_num(lon_col),
        "capacity_mw": safe_num(capacity_col),
        "status": safe_text(status_col),
        "operating_year": extract_year_series(gem_df[online_year_col]) if online_year_col else pd.Series(pd.NA, index=idx, dtype="Int64"),
        "country_name": safe_text(country_col),
    })

    country_text = out["country_name"].astype(str).str.strip().str.lower()
    status_text = out["status"].astype(str).str.strip().str.lower()

    us_values = {
        "united states",
        "united states of america",
        "usa",
        "u.s.",
        "u.s.a.",
    }

    is_us = country_text.isin(us_values) if country_col else pd.Series(True, index=out.index)
    is_operating = status_text.str.contains("operat", regex=False, na=False) if status_col else pd.Series(True, index=out.index)

    out = out[is_us & is_operating].copy()
    out = out[out["capacity_mw"].notna() & (out["capacity_mw"] > 0)].copy()

    # force these columns to exist no matter what
    for col in ["county_name", "state_name", "state_abbrev"]:
        if col not in out.columns:
            out[col] = pd.NA

    # spatial fill from lat/lon if available
    if out["longitude"].notna().any() and out["latitude"].notna().any():
        points = gpd.GeoDataFrame(
            out.copy(),
            geometry=gpd.points_from_xy(out["longitude"], out["latitude"]),
            crs="EPSG:4326"
        )

        joined = gpd.sjoin(points, counties, how="left", predicate="within")
        joined = pd.DataFrame(joined.drop(columns="geometry"))

        for col in ["county_name", "state_name", "state_abbrev"]:
            if col not in joined.columns:
                joined[col] = pd.NA

        if "county_name_geo" in joined.columns:
            joined["county_name"] = joined["county_name"].fillna(joined["county_name_geo"])
        if "state_name_geo" in joined.columns:
            joined["state_name"] = joined["state_name"].fillna(joined["state_name_geo"])
        if "state_abbrev_geo" in joined.columns:
            joined["state_abbrev"] = joined["state_abbrev"].fillna(joined["state_abbrev_geo"])

        out = joined.copy()

    # fill state abbrev from state name
    if "state_name" not in out.columns:
        out["state_name"] = pd.NA
    if "state_abbrev" not in out.columns:
        out["state_abbrev"] = pd.NA

    out = out.merge(state_lookup, on="state_name", how="left", suffixes=("", "_lookup"))
    if "state_abbrev_lookup" in out.columns:
        out["state_abbrev"] = out["state_abbrev"].fillna(out["state_abbrev_lookup"])
        out = out.drop(columns=["state_abbrev_lookup"])

    # county join key
    if "county_name" not in out.columns:
        out["county_name"] = pd.NA
    out["county_name_join"] = standardize_county_name_for_join(out["county_name"])

    county_lookup = (
        counties[["state_abbrev_geo", "county_name_join", "county_fips"]]
        .drop_duplicates()
        .rename(columns={"state_abbrev_geo": "state_abbrev"})
    )

    out = out.merge(
        county_lookup,
        on=["state_abbrev", "county_name_join"],
        how="left"
    )
    return out.reset_index(drop=True)

In [10]:
GEM_SHEET = "Utility-Scale (1 MW+)"
gem_latest = standardize_gem(GEM_FILE, gem_sheet=GEM_SHEET)

print("gem_latest shape:", gem_latest.shape)

if "county_fips" not in gem_latest.columns:
    if "county_fips_x" in gem_latest.columns and "county_fips_y" in gem_latest.columns:
        gem_latest["county_fips"] = gem_latest["county_fips_x"].fillna(gem_latest["county_fips_y"])
    elif "county_fips_x" in gem_latest.columns:
        gem_latest["county_fips"] = gem_latest["county_fips_x"]
    elif "county_fips_y" in gem_latest.columns:
        gem_latest["county_fips"] = gem_latest["county_fips_y"]

print("Missing county_name share:", gem_latest["county_name"].isna().mean())
print("Missing county_fips share:", gem_latest["county_fips"].isna().mean())

display(
    gem_latest[
        ["project_name", "country_name", "state_name", "state_abbrev", "county_name", "county_fips", "capacity_mw"]
    ].head(20)
)


GEM sheet used: Utility-Scale (1 MW+)
GEM raw shape: (103940, 32)
First 40 GEM columns:
['Date Last Researched', 'Country/Area', 'Project Name', 'Phase Name', 'Project Name in Local Language / Script', 'Other Name(s)', 'Capacity (MW)', 'Capacity Rating', 'Technology Type', 'Status', 'Start year', 'Retired year', 'Operator', 'Operator Name in Local Language / Script', 'Owner', 'Owner Name in Local Language / Script', 'Hydrogen', 'Associated Storage', 'Latitude', 'Longitude', 'Location accuracy', 'City', 'Local area (taluk, county)', 'Major area (prefecture, district)', 'State/Province', 'Subregion', 'Region', 'GEM location ID', 'GEM phase ID', 'Other IDs (location)', 'Other IDs (unit/phase)', 'Wiki URL']
Detected GEM columns:
  country: Country/Area
  status: Status
  state: State/Province
  county: Local area (taluk, county)
  lat: Latitude
  lon: Longitude
  capacity: Capacity (MW)
  project name: Project Name
  online year: Start year
  project id: GEM location ID
gem_latest shape: 

,project_name,country_name,state_name,state_abbrev,county_name,county_fips,capacity_mw
0,(3K) 59 Hetcheltown Rd solar project,United States,New York,NY,Schenectady,36093,1.4
1,0 Hammond St CSG solar farm,United States,Massachusetts,MA,Plymouth,25023,4.8
2,10 Briggs Solar NG East,United States,Rhode Island,RI,Kent,44003,5.0
3,10 Finderne Avenue Solar,United States,New Jersey,NJ,Somerset,34035,2.3
4,100 Brook Hill Drive Solar,United States,New York,NY,Rockland,36087,2.0
5,100 Lower solar farm,United States,New York,NY,Orange,36071,2.0
6,1001 Ebenezer Church Solar,United States,North Carolina,NC,Surry,37171,5.0
7,1008 Matthews Solar,United States,North Carolina,NC,Yadkin,37197,4.9
8,1009 Yadkin Solar,United States,North Carolina,NC,Yadkin,37197,4.9
9,101 Carnegie Center Solar,United States,New Jersey,NJ,Mercer,34021,1.2


In [11]:
def make_gem_snapshot(gem_df, asof_year):
    out = gem_df.copy()
    if out["operating_year"].notna().any():
        out = out[out["operating_year"].fillna(asof_year) <= asof_year].copy()
    out["snapshot_label"] = str(asof_year)
    return out

gem_2021 = make_gem_snapshot(gem_latest, 2021)
gem_2024 = make_gem_snapshot(gem_latest, 2024)

print("gem_2021 shape:", gem_2021.shape)
print("gem_2024 shape:", gem_2024.shape)

display(gem_2021.head())
display(gem_2024.head())

gem_2021 shape: (5243, 33)
gem_2024 shape: (6889, 33)


,source,source_project_id,project_name,county_name_left,state_name,state_abbrev,latitude,longitude,capacity_mw,status,operating_year,country_name,index_right,STATEFP,COUNTYFP,COUNTYNS,GEOIDFQ,county_fips_x,county_name_geo,NAMELSAD,state_abbrev_geo,state_name_geo,LSAD,ALAND,AWATER,county_name_right,full_score,cost_modifier_full,county_name_join,county_name,county_fips_y,county_fips,snapshot_label
0,GEM,L100001016961,(3K) 59 Hetcheltown Rd solar project,Schenectady County,New York,NY,42.8766,-73.9105,1.4,operating,2020,United States,2663.0,36,093,00974144,0500000US36093,36093,Schenectady,Schenectady County,NY,New York,06,5.300897e+08,1.236694e+07,None,None,None,Schenectady,Schenectady,36093,36093,2021
1,GEM,L100000811700,0 Hammond St CSG solar farm,Plymouth County,Massachusetts,MA,41.8086,-70.7267,4.8,operating,2021,United States,1341.0,25,023,00606938,0500000US25023,25023,Plymouth,Plymouth County,MA,Massachusetts,06,1.705513e+09,1.126114e+09,None,None,None,Plymouth,Plymouth,25023,25023,2021
2,GEM,L100000811701,10 Briggs Solar NG East,Kent County,Rhode Island,RI,41.6327,-71.4963,5.0,operating,2021,United States,3221.0,44,003,01219778,0500000US44003,44003,Kent,Kent County,RI,Rhode Island,06,4.365888e+08,5.068611e+07,None,None,None,Kent,Kent,44003,44003,2021
3,GEM,L100000811702,10 Finderne Avenue Solar,Somerset County,New Jersey,NJ,40.5581,-74.5759,2.3,operating,2017,United States,582.0,34,035,00882234,0500000US34035,34035,Somerset,Somerset County,NJ,New Jersey,06,7.818321e+08,7.993021e+06,None,None,None,Somerset,Somerset,34035,34035,2021
4,GEM,L100000811703,100 Brook Hill Drive Solar,Rockland County,New York,NY,41.0931,-73.9828,2.0,operating,2015,United States,2665.0,36,087,00974142,0500000US36087,36087,Rockland,Rockland County,NY,New York,06,4.498410e+08,6.691967e+07,None,None,None,Rockland,Rockland,36087,36087,2021


,source,source_project_id,project_name,county_name_left,state_name,state_abbrev,latitude,longitude,capacity_mw,status,operating_year,country_name,index_right,STATEFP,COUNTYFP,COUNTYNS,GEOIDFQ,county_fips_x,county_name_geo,NAMELSAD,state_abbrev_geo,state_name_geo,LSAD,ALAND,AWATER,county_name_right,full_score,cost_modifier_full,county_name_join,county_name,county_fips_y,county_fips,snapshot_label
0,GEM,L100001016961,(3K) 59 Hetcheltown Rd solar project,Schenectady County,New York,NY,42.8766,-73.9105,1.4,operating,2020,United States,2663.0,36,093,00974144,0500000US36093,36093,Schenectady,Schenectady County,NY,New York,06,5.300897e+08,1.236694e+07,None,None,None,Schenectady,Schenectady,36093,36093,2024
1,GEM,L100000811700,0 Hammond St CSG solar farm,Plymouth County,Massachusetts,MA,41.8086,-70.7267,4.8,operating,2021,United States,1341.0,25,023,00606938,0500000US25023,25023,Plymouth,Plymouth County,MA,Massachusetts,06,1.705513e+09,1.126114e+09,None,None,None,Plymouth,Plymouth,25023,25023,2024
2,GEM,L100000811701,10 Briggs Solar NG East,Kent County,Rhode Island,RI,41.6327,-71.4963,5.0,operating,2021,United States,3221.0,44,003,01219778,0500000US44003,44003,Kent,Kent County,RI,Rhode Island,06,4.365888e+08,5.068611e+07,None,None,None,Kent,Kent,44003,44003,2024
3,GEM,L100000811702,10 Finderne Avenue Solar,Somerset County,New Jersey,NJ,40.5581,-74.5759,2.3,operating,2017,United States,582.0,34,035,00882234,0500000US34035,34035,Somerset,Somerset County,NJ,New Jersey,06,7.818321e+08,7.993021e+06,None,None,None,Somerset,Somerset,34035,34035,2024
4,GEM,L100000811703,100 Brook Hill Drive Solar,Rockland County,New York,NY,41.0931,-73.9828,2.0,operating,2015,United States,2665.0,36,087,00974142,0500000US36087,36087,Rockland,Rockland County,NY,New York,06,4.498410e+08,6.691967e+07,None,None,None,Rockland,Rockland,36087,36087,2024


In [12]:
# Clean GEM county/state columns after joins
for df_name in ["gem_latest", "gem_2021", "gem_2024"]:
    df = globals()[df_name].copy()

    if "county_name" not in df.columns:
        if "county_name_left" in df.columns and "county_name_right" in df.columns:
            df["county_name"] = df["county_name_left"].fillna(df["county_name_right"])
        elif "county_name_left" in df.columns:
            df["county_name"] = df["county_name_left"]
        elif "county_name_right" in df.columns:
            df["county_name"] = df["county_name_right"]

    if "county_fips" not in df.columns:
        if "county_fips_x" in df.columns and "county_fips_y" in df.columns:
            df["county_fips"] = df["county_fips_x"].fillna(df["county_fips_y"])
        elif "county_fips_x" in df.columns:
            df["county_fips"] = df["county_fips_x"]
        elif "county_fips_y" in df.columns:
            df["county_fips"] = df["county_fips_y"]

    globals()[df_name] = df

print("gem_latest missing county_name share:", gem_latest["county_name"].isna().mean())
print("gem_latest missing county_fips share:", gem_latest["county_fips"].isna().mean())
print("gem_2021 missing county_fips share:", gem_2021["county_fips"].isna().mean())
print("gem_2024 missing county_fips share:", gem_2024["county_fips"].isna().mean())

gem_latest missing county_name share: 0.0002760143527463428
gem_latest missing county_fips share: 0.0002760143527463428
gem_2021 missing county_fips share: 0.0
gem_2024 missing county_fips share: 0.00029031789809841774


In [13]:
eia_2021_county = aggregate_capacity(eia_2021, ["state_abbrev", "state_name", "county_name", "county_fips"])
eia_2024_county = aggregate_capacity(eia_2024, ["state_abbrev", "state_name", "county_name", "county_fips"])

gem_2021_county = aggregate_capacity(gem_2021, ["state_abbrev", "state_name", "county_name", "county_fips"])
gem_2024_county = aggregate_capacity(gem_2024, ["state_abbrev", "state_name", "county_name", "county_fips"])

eia_2021_state = aggregate_capacity(eia_2021, ["state_abbrev", "state_name"])
eia_2024_state = aggregate_capacity(eia_2024, ["state_abbrev", "state_name"])

gem_2021_state = aggregate_capacity(gem_2021, ["state_abbrev", "state_name"])
gem_2024_state = aggregate_capacity(gem_2024, ["state_abbrev", "state_name"])

display(eia_2021_state.head(20))
display(eia_2024_state.head(20))
display(gem_2021_state.head(20))
display(gem_2024_state.head(20))

,state_abbrev,state_name,project_count,total_capacity_mw,avg_capacity_mw
4,CA,California,954,15577.1,16.328197
42,TX,Texas,111,8837.7,79.618919
27,NC,North Carolina,724,5744.7,7.934669
9,FL,Florida,102,4953.0,48.558824
32,NV,Nevada,77,3150.6,40.916883
10,GA,Georgia,110,3080.0,28.000000
3,AZ,Arizona,132,2810.6,21.292424
44,VA,Virginia,63,2203.1,34.969841
43,UT,Utah,39,1457.2,37.364103
19,MA,Massachusetts,460,1195.1,2.598043


,state_abbrev,state_name,project_count,total_capacity_mw,avg_capacity_mw
4,CA,California,1044,22558.3,21.607567
42,TX,Texas,186,22465.6,120.782796
9,FL,Florida,193,10951.9,56.745596
27,NC,North Carolina,793,6783.1,8.553720
3,AZ,Arizona,147,5336.2,36.300680
32,NV,Nevada,102,5285.1,51.814706
10,GA,Georgia,153,5013.4,32.767320
44,VA,Virginia,130,4822.7,37.097692
34,OH,Ohio,63,3238.0,51.396825
14,IL,Illinois,218,2953.6,13.548624


,state_abbrev,state_name,project_count,total_capacity_mw,avg_capacity_mw
3,CA,California,813,15430.2,18.979336
41,TX,Texas,115,8957.0,77.886957
26,NC,North Carolina,709,5843.8,8.242313
8,FL,Florida,105,4958.1,47.220000
9,GA,Georgia,105,3051.8,29.064762
31,NV,Nevada,75,2983.6,39.781333
2,AZ,Arizona,108,2779.0,25.731481
43,VA,Virginia,66,2267.1,34.350000
42,UT,Utah,37,1458.0,39.405405
18,MA,Massachusetts,482,1287.3,2.670747


,state_abbrev,state_name,project_count,total_capacity_mw,avg_capacity_mw
43,TX,Texas,195,25073.3,128.581026
4,CA,California,983,22588.8,22.979451
9,FL,Florida,195,11027.1,56.549231
27,NC,North Carolina,760,6767.8,8.905000
3,AZ,Arizona,126,5302.6,42.084127
32,NV,Nevada,100,5297.6,52.976000
10,GA,Georgia,149,5011.2,33.632215
45,VA,Virginia,128,4856.2,37.939062
35,OH,Ohio,63,3086.6,48.993651
14,IL,Illinois,215,2956.5,13.751163


In [14]:
compare_state_2021 = compare_aggregates(
    eia_2021_state, gem_2021_state,
    ["state_abbrev", "state_name"],
    "eia_2021", "gem_2021"
)

compare_state_2024 = compare_aggregates(
    eia_2024_state, gem_2024_state,
    ["state_abbrev", "state_name"],
    "eia_2024", "gem_2024"
)

compare_county_2021 = compare_aggregates(
    eia_2021_county, gem_2021_county,
    ["state_abbrev", "state_name", "county_name", "county_fips"],
    "eia_2021", "gem_2021"
)

compare_county_2024 = compare_aggregates(
    eia_2024_county, gem_2024_county,
    ["state_abbrev", "state_name", "county_name", "county_fips"],
    "eia_2024", "gem_2024"
)

display(compare_state_2021.head(20))
display(compare_state_2024.head(20))
display(compare_county_2021.head(20))
display(compare_county_2024.head(20))

,state_abbrev,state_name,project_count_eia_2021,total_capacity_mw_eia_2021,avg_capacity_mw_eia_2021,project_count_gem_2021,total_capacity_mw_gem_2021,avg_capacity_mw_gem_2021,capacity_diff_mw_eia_2021_minus_gem_2021,project_count_diff_eia_2021_minus_gem_2021
0,AK,Alaska,8,0.8,0.100000,NaN,NaN,NaN,0.8,8.0
1,AL,Alabama,7,423.9,60.557143,8.0,453.9,56.737500,-30.0,-1.0
2,AR,Arkansas,15,223.7,14.913333,33.0,245.4,7.436364,-21.7,-18.0
3,AZ,Arizona,132,2810.6,21.292424,108.0,2779.0,25.731481,31.6,24.0
4,CA,California,954,15577.1,16.328197,813.0,15430.2,18.979336,146.9,141.0
5,CO,Colorado,115,1060.9,9.225217,125.0,1107.5,8.860000,-46.6,-10.0
6,CT,Connecticut,61,245.8,4.029508,66.0,263.8,3.996970,-18.0,-5.0
7,DC,District of Columbia,7,15.3,2.185714,8.0,18.0,2.250000,-2.7,-1.0
8,DE,Delaware,12,38.1,3.175000,11.0,37.6,3.418182,0.5,1.0
9,FL,Florida,102,4953.0,48.558824,105.0,4958.1,47.220000,-5.1,-3.0


,state_abbrev,state_name,project_count_eia_2024,total_capacity_mw_eia_2024,avg_capacity_mw_eia_2024,project_count_gem_2024,total_capacity_mw_gem_2024,avg_capacity_mw_gem_2024,capacity_diff_mw_eia_2024_minus_gem_2024,project_count_diff_eia_2024_minus_gem_2024
0,AK,Alaska,18.0,7.7,0.427778,1.0,6.0,6.000000,1.700000e+00,17.0
1,AL,Alabama,11.0,666.3,60.572727,11.0,666.3,60.572727,0.000000e+00,0.0
2,AR,Arkansas,62.0,1793.7,28.930645,65.0,1821.7,28.026154,-2.800000e+01,-3.0
3,AZ,Arizona,147.0,5336.2,36.300680,126.0,5302.6,42.084127,3.360000e+01,21.0
4,CA,California,1044.0,22558.3,21.607567,983.0,22588.8,22.979451,-3.050000e+01,61.0
5,CO,Colorado,166.0,2371.2,14.284337,169.0,2396.9,14.182840,-2.570000e+01,-3.0
6,CT,Connecticut,85.0,322.8,3.797647,80.0,321.4,4.017500,1.400000e+00,5.0
7,DC,District of Columbia,15.0,29.2,1.946667,12.0,26.1,2.175000,3.100000e+00,3.0
8,DE,Delaware,17.0,97.3,5.723529,16.0,96.8,6.050000,5.000000e-01,1.0
9,FL,Florida,193.0,10951.9,56.745596,195.0,11027.1,56.549231,-7.520000e+01,-2.0


,state_abbrev,state_name,county_name,county_fips,project_count_eia_2021,total_capacity_mw_eia_2021,avg_capacity_mw_eia_2021,project_count_gem_2021,total_capacity_mw_gem_2021,avg_capacity_mw_gem_2021,capacity_diff_mw_eia_2021_minus_gem_2021,project_count_diff_eia_2021_minus_gem_2021
0,AK,Alaska,NORTHWEST ARCTIC,NaN,8.0,0.8,0.1,NaN,NaN,NaN,0.8,8.0
1,AL,Alabama,Calhoun,01015,1.0,7.4,7.4,1.0,7.4,7.4,0.0,0.0
2,AL,Alabama,Chambers,01017,1.0,79.2,79.2,1.0,79.2,79.2,0.0,0.0
3,AL,Alabama,Colbert,01033,1.0,227.0,227.0,1.0,227.0,227.0,0.0,0.0
4,AL,Alabama,Dale,01045,1.0,10.6,10.6,1.0,10.6,10.6,0.0,0.0
5,AL,Alabama,Lauderdale,01077,1.0,75.0,75.0,1.0,75.0,75.0,0.0,0.0
6,AL,Alabama,Limestone,01083,1.0,14.7,14.7,1.0,14.7,14.7,0.0,0.0
7,AL,Alabama,Madison,01089,1.0,10.0,10.0,1.0,10.0,10.0,0.0,0.0
8,AL,Alabama,Russell,01113,NaN,NaN,NaN,1.0,30.0,30.0,-30.0,-1.0
9,AR,Arkansas,Arkansas,05001,1.0,81.0,81.0,1.0,81.0,81.0,0.0,0.0


,state_abbrev,state_name,county_name,county_fips,project_count_eia_2024,total_capacity_mw_eia_2024,avg_capacity_mw_eia_2024,project_count_gem_2024,total_capacity_mw_gem_2024,avg_capacity_mw_gem_2024,capacity_diff_mw_eia_2024_minus_gem_2024,project_count_diff_eia_2024_minus_gem_2024
0,AK,Alaska,Matanuska Susitna,NaN,1.0,6.0,6.0,NaN,NaN,NaN,6.0,1.0
1,AK,Alaska,Matanuska-Susitna,02170,NaN,NaN,NaN,1.0,6.0,6.0,-6.0,-1.0
2,AK,Alaska,NORTHWEST ARCTIC,NaN,17.0,1.7,0.1,NaN,NaN,NaN,1.7,17.0
3,AL,Alabama,Calhoun,01015,1.0,7.4,7.4,1.0,7.4,7.4,0.0,0.0
4,AL,Alabama,Chambers,01017,1.0,79.2,79.2,1.0,79.2,79.2,0.0,0.0
5,AL,Alabama,Colbert,01033,1.0,227.0,227.0,1.0,227.0,227.0,0.0,0.0
6,AL,Alabama,Covington,01039,1.0,80.0,80.0,1.0,80.0,80.0,0.0,0.0
7,AL,Alabama,Dale,01045,1.0,10.6,10.6,1.0,10.6,10.6,0.0,0.0
8,AL,Alabama,Lauderdale,01077,1.0,75.0,75.0,1.0,75.0,75.0,0.0,0.0
9,AL,Alabama,Limestone,01083,1.0,14.7,14.7,1.0,14.7,14.7,0.0,0.0


In [15]:
eia_2021.to_csv(OUTPUT_DIR / "eia_2021_solar_projects_clean.csv", index=False)
eia_2024.to_csv(OUTPUT_DIR / "eia_2024_solar_projects_clean.csv", index=False)

gem_2021.to_csv(OUTPUT_DIR / "gem_2021_solar_projects_clean.csv", index=False)
gem_2024.to_csv(OUTPUT_DIR / "gem_2024_solar_projects_clean.csv", index=False)

eia_2021_county.to_csv(OUTPUT_DIR / "eia_2021_solar_by_county.csv", index=False)
eia_2024_county.to_csv(OUTPUT_DIR / "eia_2024_solar_by_county.csv", index=False)

eia_2021_state.to_csv(OUTPUT_DIR / "eia_2021_solar_by_state.csv", index=False)
eia_2024_state.to_csv(OUTPUT_DIR / "eia_2024_solar_by_state.csv", index=False)

gem_2021_county.to_csv(OUTPUT_DIR / "gem_2021_solar_by_county.csv", index=False)
gem_2024_county.to_csv(OUTPUT_DIR / "gem_2024_solar_by_county.csv", index=False)

gem_2021_state.to_csv(OUTPUT_DIR / "gem_2021_solar_by_state.csv", index=False)
gem_2024_state.to_csv(OUTPUT_DIR / "gem_2024_solar_by_state.csv", index=False)

compare_state_2021.to_csv(OUTPUT_DIR / "compare_eia_vs_gem_state_2021.csv", index=False)
compare_state_2024.to_csv(OUTPUT_DIR / "compare_eia_vs_gem_state_2024.csv", index=False)

compare_county_2021.to_csv(OUTPUT_DIR / "compare_eia_vs_gem_county_2021.csv", index=False)
compare_county_2024.to_csv(OUTPUT_DIR / "compare_eia_vs_gem_county_2024.csv", index=False)

print("Saved files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)

Saved files:
 - compare_eia_vs_gem_county_2021.csv
 - compare_eia_vs_gem_county_2024.csv
 - compare_eia_vs_gem_state_2021.csv
 - compare_eia_vs_gem_state_2024.csv
 - eia_2021_solar_by_county.csv
 - eia_2021_solar_by_state.csv
 - eia_2021_solar_projects_clean.csv
 - eia_2024_solar_by_county.csv
 - eia_2024_solar_by_state.csv
 - eia_2024_solar_projects_clean.csv
 - gem_2021_solar_by_county.csv
 - gem_2021_solar_by_state.csv
 - gem_2021_solar_projects_clean.csv
 - gem_2024_solar_by_county.csv
 - gem_2024_solar_by_state.csv
 - gem_2024_solar_projects_clean.csv
 - run_summary.json


In [16]:
print("Top EIA 2021 states")
display(eia_2021_state.head(15))

print("Top EIA 2024 states")
display(eia_2024_state.head(15))

print("Top GEM 2021 states")
display(gem_2021_state.head(15))

print("Top GEM 2024 states")
display(gem_2024_state.head(15))

Top EIA 2021 states


,state_abbrev,state_name,project_count,total_capacity_mw,avg_capacity_mw
4,CA,California,954,15577.1,16.328197
42,TX,Texas,111,8837.7,79.618919
27,NC,North Carolina,724,5744.7,7.934669
9,FL,Florida,102,4953.0,48.558824
32,NV,Nevada,77,3150.6,40.916883
10,GA,Georgia,110,3080.0,28.000000
3,AZ,Arizona,132,2810.6,21.292424
44,VA,Virginia,63,2203.1,34.969841
43,UT,Utah,39,1457.2,37.364103
19,MA,Massachusetts,460,1195.1,2.598043


Top EIA 2024 states


,state_abbrev,state_name,project_count,total_capacity_mw,avg_capacity_mw
4,CA,California,1044,22558.3,21.607567
42,TX,Texas,186,22465.6,120.782796
9,FL,Florida,193,10951.9,56.745596
27,NC,North Carolina,793,6783.1,8.553720
3,AZ,Arizona,147,5336.2,36.300680
32,NV,Nevada,102,5285.1,51.814706
10,GA,Georgia,153,5013.4,32.767320
44,VA,Virginia,130,4822.7,37.097692
34,OH,Ohio,63,3238.0,51.396825
14,IL,Illinois,218,2953.6,13.548624


Top GEM 2021 states


,state_abbrev,state_name,project_count,total_capacity_mw,avg_capacity_mw
3,CA,California,813,15430.2,18.979336
41,TX,Texas,115,8957.0,77.886957
26,NC,North Carolina,709,5843.8,8.242313
8,FL,Florida,105,4958.1,47.220000
9,GA,Georgia,105,3051.8,29.064762
31,NV,Nevada,75,2983.6,39.781333
2,AZ,Arizona,108,2779.0,25.731481
43,VA,Virginia,66,2267.1,34.350000
42,UT,Utah,37,1458.0,39.405405
18,MA,Massachusetts,482,1287.3,2.670747


Top GEM 2024 states


,state_abbrev,state_name,project_count,total_capacity_mw,avg_capacity_mw
43,TX,Texas,195,25073.3,128.581026
4,CA,California,983,22588.8,22.979451
9,FL,Florida,195,11027.1,56.549231
27,NC,North Carolina,760,6767.8,8.905000
3,AZ,Arizona,126,5302.6,42.084127
32,NV,Nevada,100,5297.6,52.976000
10,GA,Georgia,149,5011.2,33.632215
45,VA,Virginia,128,4856.2,37.939062
35,OH,Ohio,63,3086.6,48.993651
14,IL,Illinois,215,2956.5,13.751163


In [17]:
print("Largest state capacity differences, 2021")
display(
    compare_state_2021.sort_values("capacity_diff_mw_eia_2021_minus_gem_2021", ascending=False).head(20)
)

print("Largest state capacity differences, 2024")
display(
    compare_state_2024.sort_values("capacity_diff_mw_eia_2024_minus_gem_2024", ascending=False).head(20)
)

Largest state capacity differences, 2021


,state_abbrev,state_name,project_count_eia_2021,total_capacity_mw_eia_2021,avg_capacity_mw_eia_2021,project_count_gem_2021,total_capacity_mw_gem_2021,avg_capacity_mw_gem_2021,capacity_diff_mw_eia_2021_minus_gem_2021,project_count_diff_eia_2021_minus_gem_2021
32,NV,Nevada,77,3150.6,40.916883,75.0,2983.6,39.781333,167.0,2.0
4,CA,California,954,15577.1,16.328197,813.0,15430.2,18.979336,146.9,141.0
3,AZ,Arizona,132,2810.6,21.292424,108.0,2779.0,25.731481,31.6,24.0
10,GA,Georgia,110,3080.0,28.000000,105.0,3051.8,29.064762,28.2,5.0
11,HI,Hawaii,48,288.9,6.018750,23.0,281.2,12.226087,7.7,25.0
41,TN,Tennessee,20,194.0,9.700000,19.0,191.3,10.068421,2.7,1.0
22,MI,Michigan,40,354.4,8.860000,37.0,352.6,9.529730,1.8,3.0
28,NE,Nebraska,13,33.2,2.553846,11.0,31.8,2.890909,1.4,2.0
46,WA,Washington,4,23.4,5.850000,2.0,22.4,11.200000,1.0,2.0
0,AK,Alaska,8,0.8,0.100000,NaN,NaN,NaN,0.8,8.0


Largest state capacity differences, 2024


,state_abbrev,state_name,project_count_eia_2024,total_capacity_mw_eia_2024,avg_capacity_mw_eia_2024,project_count_gem_2024,total_capacity_mw_gem_2024,avg_capacity_mw_gem_2024,capacity_diff_mw_eia_2024_minus_gem_2024,project_count_diff_eia_2024_minus_gem_2024
35,OH,Ohio,63.0,3238.0,51.396825,63.0,3086.6,48.993651,151.4,0.0
34,NY,New York,632.0,2674.6,4.231962,623.0,2638.7,4.235474,35.9,9.0
3,AZ,Arizona,147.0,5336.2,36.300680,126.0,5302.6,42.084127,33.6,21.0
21,ME,Maine,141.0,832.9,5.907092,133.0,805.6,6.057143,27.3,8.0
27,NC,North Carolina,793.0,6783.1,8.553720,760.0,6767.8,8.905000,15.3,33.0
40,SC,South Carolina,122.0,1649.2,13.518033,121.0,1641.9,13.569421,7.3,1.0
38,PA,Pennsylvania,65.0,876.0,13.476923,60.0,868.9,14.481667,7.1,5.0
19,MA,Massachusetts,544.0,1432.3,2.632904,524.0,1425.5,2.720420,6.8,20.0
7,DC,District of Columbia,15.0,29.2,1.946667,12.0,26.1,2.175000,3.1,3.0
10,GA,Georgia,153.0,5013.4,32.767320,149.0,5011.2,33.632215,2.2,4.0


In [18]:
summary = {
    "eia_2021_rows": int(len(eia_2021)),
    "eia_2024_rows": int(len(eia_2024)),
    "gem_latest_rows": int(len(gem_latest)),
    "gem_2021_rows": int(len(gem_2021)),
    "gem_2024_rows": int(len(gem_2024)),
}

with open(OUTPUT_DIR / "run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

summary

{'eia_2021_rows': 5270,
 'eia_2024_rows': 7161,
 'gem_latest_rows': 7246,
 'gem_2021_rows': 5243,
 'gem_2024_rows': 6889}

State-level EIA and GEM solar capacity comparisons for 2021 and 2024 are complete and closely aligned. County-level outputs are also complete for the large majority of records, with a small number of remaining unmatched county-equivalent geographies caused by naming differences (for example, Alaska borough/census-area conventions).